# ARC-04 Units Tests

Executable notebook version of `test_units.py`.


In [ ]:
import pytest

from quantum_space_agi.scientific.errors import (
    ScientificError,
    ScientificErrorCode,
)
from quantum_space_agi.scientific.units import (
    Quantity,
    Unit,
    convert,
    get_unit,
    is_compatible,
)


def test_get_registered_unit():
    assert get_unit("km").symbol == "km"
    assert get_unit("m").scale_to_si == 1.0


def test_convert_kilometers_to_meters():
    result = convert(Quantity(5, get_unit("km")), "m")
    assert result.value == pytest.approx(5000)
    assert result.unit.symbol == "m"


def test_convert_kilometers_per_hour_to_meters_per_second():
    result = convert(Quantity(36, get_unit("km/h")), "m/s")
    assert result.value == pytest.approx(10)


def test_convert_hours_to_seconds():
    result = convert(Quantity(2, get_unit("h")), "s")
    assert result.value == pytest.approx(7200)


def test_compatible_units():
    assert is_compatible("km", "m")
    assert is_compatible("h", "s")
    assert is_compatible("km/h", "m/s")


def test_incompatible_units():
    assert not is_compatible("km", "s")
    assert not is_compatible("kg", "N")


def test_conversion_rejects_incompatible_units():
    with pytest.raises(ScientificError) as exc:
        convert(Quantity(5, get_unit("km")), "s")

    assert exc.value.code == ScientificErrorCode.UNIT_MISMATCH


def test_unknown_unit_is_rejected():
    with pytest.raises(ScientificError):
        get_unit("lightyear")


@pytest.mark.parametrize("value", [float("nan"), float("inf"), float("-inf"), True])
def test_quantity_rejects_invalid_values(value):
    with pytest.raises(ScientificError):
        Quantity(value, get_unit("m"))


def test_unit_is_immutable():
    unit = get_unit("m")

    with pytest.raises(AttributeError):
        unit.symbol = "km"


def test_quantity_is_immutable():
    quantity = Quantity(10, get_unit("m"))

    with pytest.raises(AttributeError):
        quantity.value = 20


def test_custom_unit():
    custom = Unit("millimeter", 0.001, (1, 0, 0, 0, 0, 0, 0))
    result = convert(Quantity(1000, custom), "m")

    assert result.value == pytest.approx(1)


In [ ]:
import inspect
import itertools

executed_test_cases = 0

for test_name, test_function in sorted(list(globals().items())):
    if not test_name.startswith("test_") or not callable(test_function):
        continue

    parameter_marks = [
        mark for mark in getattr(test_function, "pytestmark", [])
        if getattr(mark, "name", "") == "parametrize"
    ]

    parameter_sets = [{}]
    for mark in parameter_marks:
        argnames = mark.args[0]
        argvalues = mark.args[1]
        if isinstance(argnames, str):
            names = [name.strip() for name in argnames.split(",")]
        else:
            names = list(argnames)

        next_sets = []
        for existing in parameter_sets:
            for argvalue in argvalues:
                if len(names) == 1:
                    arguments = {names[0]: argvalue}
                elif isinstance(argvalue, tuple) or isinstance(argvalue, list):
                    arguments = dict(zip(names, argvalue))
                else:
                    raise TypeError(
                        f"{test_name} parameterization requires one value per name"
                    )
                next_sets.append({**existing, **arguments})
        parameter_sets = next_sets

    if not parameter_marks:
        parameter_sets = [{}]

    for arguments in parameter_sets:
        test_function(**arguments)
        executed_test_cases += 1

print(f"ARC-04 notebook test cases executed: {executed_test_cases}")
assert executed_test_cases > 0
